<a href="https://colab.research.google.com/github/Anshitha9286/Capstone-project/blob/main/sales_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sales Analytics Platform: Data Processing and EDA
**Dataset:** Online Retail II (UK online retailer, Dec 2009 to Dec 2011)
**Goal:** Understand, clean and explore the data before building forecasting and customer models.

## 1. Import libraries
Libraries are ready-made toolkits. We import them once at the top so every later step can use them.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 2. Load the dataset
Download the file from Kaggle, then upload it here. We read it into a pandas DataFrame, which is a table with rows and columns.

In [5]:
from google.colab import files
uploaded = files.upload()

Saving online_retail_II.csv to online_retail_II.csv


In [6]:
print(uploaded.keys())

dict_keys(['online_retail_II.csv'])


## 3. Read the data into pandas
CSV and Excel files are read differently. The original dataset has two sheets (2009-2010 and 2010-2011), so for Excel we read both and join them.

In [7]:
df = pd.read_csv("online_retail_II.csv")

In [8]:
print(df.shape)

(1067371, 8)


## 4. First look at the data
Before cleaning anything, we check what each row means, what each column contains, and what types of data we have.

In [9]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [11]:
df.describe()

,Quantity,Price,Customer ID
count,"1,067,371.00","1,067,371.00","824,364.00"
mean,9.94,4.65,"15,324.64"
std,172.71,123.55,"1,697.46"
min,"-80,995.00","-53,594.36","12,346.00"
25%,1.00,1.25,"13,975.00"
50%,3.00,2.10,"15,255.00"
75%,10.00,4.15,"16,797.00"
max,"80,995.00","38,970.00","18,287.00"


## 5. Standardize column names
Different versions of this dataset use different column names (Invoice vs InvoiceNo, Price vs UnitPrice, Customer ID vs CustomerID). We rename them so the rest of the code always uses the same names.

In [12]:
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={"InvoiceNo": "Invoice", "UnitPrice": "Price", "Customer ID": "CustomerID"})
print(df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']


## 6. Fix data types
Wrong data types cause silent mistakes. Dates stored as text cannot be grouped by month, and customer IDs stored as decimals (like 13085.0) cause problems when joining tables later.

In [13]:
df["Invoice"]     = df["Invoice"].astype(str).str.strip()
df["StockCode"]   = df["StockCode"].astype(str).str.strip().str.upper()
df["Description"] = df["Description"].astype("string").str.strip()
df["Country"]     = df["Country"].astype(str).str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"]  = pd.to_numeric(df["CustomerID"], errors="coerce").astype("Int64")
print(df.dtypes)

Invoice                object
StockCode              object
Description    string[python]
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
CustomerID              Int64
Country                object
dtype: object


## 7. Dataset overview
Time period covered, and how many invoices, products, customers and countries the data contains.

In [14]:
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Invoices  :", df["Invoice"].nunique())
print("Products  :", df["StockCode"].nunique())
print("Customers :", df["CustomerID"].nunique())
print("Countries :", df["Country"].nunique())

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Invoices  : 53628
Products  : 5131
Customers : 5942
Countries : 43


In [15]:
df.describe()

,Quantity,InvoiceDate,Price,CustomerID
count,"1,067,371.00",1067371,"1,067,371.00","824,364.00"
mean,9.94,2011-01-02 21:13:55.394028544,4.65,"15,324.64"
min,"-80,995.00",2009-12-01 07:45:00,"-53,594.36","12,346.00"
25%,1.00,2010-07-09 09:46:00,1.25,"13,975.00"
50%,3.00,2010-12-07 15:28:00,2.10,"15,255.00"
75%,10.00,2011-07-22 10:23:00,4.15,"16,797.00"
max,"80,995.00",2011-12-09 12:50:00,"38,970.00","18,287.00"
std,172.71,NaN,123.55,"1,697.46"


**Observations from df.describe()**
- **Size and period:** the dataset has 1,067,371 rows covering 1 Dec 2009 to 9 Dec 2011.
- **Missing customers:** Customer ID has 824,364 values, so 243,007 rows (about 22.8%) have no customer ID. These sales cannot be linked to a customer.
- **Quantity:** ranges from -80,995 to 80,995. Negative values are returns or cancellations. The median is 3 units but the mean is 9.94, so a few very large orders pull the average up.
- **Price:** ranges from -53,594.36 to 38,970.00. A negative price is not a real sale, so these rows are probably accounting adjustments. The median price is £2.10, so the 38,970 maximum is an extreme value that needs checking.
- **Incomplete last month:** the last date is 9 Dec 2011, so December 2011 is only a partial month. It must be excluded from monthly trend charts, or it will look like a sales crash.
- **Conclusion:** the data needs cleaning before analysis. The next step is a data quality audit to measure how many rows each problem affects.

## 8. Data quality audit
We measure each data problem before fixing anything. This lets us say exactly how many rows each issue affects and justify every cleaning decision.

### 8.1 Missing values

In [16]:
missing = df.isna().sum().to_frame("missing")
missing["pct"] = (missing["missing"] / len(df) * 100).round(2)
missing

,missing,pct
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
CustomerID,243007,22.77
Country,0,0.00


### 8.2 Duplicate rows
A duplicate is a row identical to another row in every column. Duplicates would count the same sale twice and inflate revenue.

In [17]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 34335


### 8.3 Cancellation invoices
Invoice numbers starting with "C" are cancellations or returns, not sales. If they stay in, revenue will be understated.

In [18]:
is_cancel = df["Invoice"].str.startswith("C")
print("Cancellation rows:", is_cancel.sum())
df[is_cancel].head()

Cancellation rows: 19494


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia


### 8.4 Invalid quantities and prices
A real sale has a positive quantity and a positive price. Anything else is a return, a stock adjustment or an accounting entry.

In [19]:
print("Quantity <= 0 :", (df["Quantity"] <= 0).sum())
print("Price <= 0    :", (df["Price"] <= 0).sum())

Quantity <= 0 : 22950
Price <= 0    : 6207


In [20]:
df[df["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,"-53,594.36",<NA>,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,"-44,031.79",<NA>,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,"-38,925.87",<NA>,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,"-11,062.06",<NA>,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,"-11,062.06",<NA>,United Kingdom


In [21]:
df.sort_values("Quantity", ascending=False).head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902,Denmark


### 8.5 Non-product stock codes
Real products have codes that start with 5 digits (like 85123A). Other codes, such as postage or bank charges, are fees and manual entries. They are not products and would distort product analysis.

In [22]:
non_product = ~df["StockCode"].str.match(r"^\d{5}")
print("Non-product rows:", non_product.sum())
df.loc[non_product, "StockCode"].value_counts().head(10)

Non-product rows: 6093


,count
StockCode,
POST,2122
DOT,1446
M,1426
C2,282
D,177
S,104
BANK CHARGES,102
ADJUST,67
AMAZONFEE,43


In [24]:
top_codes = df.loc[non_product, "StockCode"].value_counts().head(10).index
df.loc[non_product].groupby("StockCode")["Description"].first().loc[top_codes]

,Description
StockCode,
POST,POSTAGE
DOT,DOTCOM POSTAGE
M,Manual
C2,CARRIAGE
D,Discount
S,SAMPLES
BANK CHARGES,Bank Charges
ADJUST,Adjustment by john on 26/01/2010 16
AMAZONFEE,AMAZON FEE


### 8.6 Audit summary
All the problems in one table. A row can appear in more than one issue (for example a cancellation also has a negative quantity), so the rows do not add up to a total.

In [25]:
n = len(df)
audit = pd.DataFrame({
    "issue": ["Missing Customer ID", "Missing Description", "Duplicate rows",
              "Cancellation invoices (C...)", "Quantity <= 0", "Price <= 0",
              "Non-product stock codes"],
    "rows": [df["CustomerID"].isna().sum(), df["Description"].isna().sum(),
             df.duplicated().sum(), is_cancel.sum(),
             (df["Quantity"] <= 0).sum(), (df["Price"] <= 0).sum(),
             non_product.sum()],
})
audit["% of rows"] = (audit["rows"] / n * 100).round(2)
audit

,issue,rows,% of rows
0,Missing Customer ID,243007,22.77
1,Missing Description,4382,0.41
2,Duplicate rows,34335,3.22
3,Cancellation invoices (C...),19494,1.83
4,Quantity <= 0,22950,2.15
5,Price <= 0,6207,0.58
6,Non-product stock codes,6093,0.57


**Observations from the audit**
- **Missing Customer ID is the largest issue:** 243,007 rows (22.77%). These are real sales, so they are kept for revenue analysis and excluded only from customer analysis (RFM, churn).
- **Duplicates:** 34,335 rows (3.22%) are identical in every column. Assumption: an identical row on the same invoice is a double entry, so they will be dropped.
- **Cancellations:** 19,494 rows (1.83%) have invoice numbers starting with C. They will be moved to a separate returns table.
- **Quantity <= 0 affects 22,950 rows, more than the 19,494 cancellations.** So about 3,456 rows have zero or negative quantity without being cancellations, probably stock adjustments.
- **Price <= 0 affects 6,207 rows.** Only 5 have a negative price. All five are "Adjust bad debt" entries (stock code B, invoices starting with A, no customer ID) from -£11,062 to -£53,594. They are accounting entries, not sales.
- **Non-product stock codes:** 6,093 rows (0.57%). The most common are POST, DOT, M, C2, D, S, BANK CHARGES, ADJUST and AMAZONFEE, which look like postage, manual entries, fees and discounts, not products.
- **Extreme quantities:** the largest line is 80,995 units (PAPER CRAFT, LITTLE BIRDIE) and the smallest quantity in the data is -80,995. The same size suggests an order that was cancelled, which needs checking.
- **Conclusion:** none of these problems is deleted blindly. Each one is moved to its own table or handled with a logged decision.

## 9. Data cleaning
Every cleaning step is logged. Problems are moved to separate tables instead of deleted, so nothing is lost and every decision can be explained.

| Problem | Decision | Reason |
|---|---|---|
| Duplicate rows | Drop | Counting the same sale twice inflates revenue |
| Cancellations (C...) | Move to a `returns` table | Returns are useful to analyze but are not sales |
| Quantity or Price <= 0 | Move to an `adjustments` table | Stock write-offs and accounting entries, not sales |
| Non-product codes | Move to a `non_products` table | Postage and fees are not products |
| Missing Description | Fill from the same stock code | A product keeps one name |
| Missing Customer ID | Keep | Still a valid sale |

In [26]:
work = df.copy()
log = []

def record(step, before, after):
    log.append({"step": step, "rows_before": before,
                "rows_after": after, "rows_removed": before - after})

In [27]:
b = len(work)
work = work.drop_duplicates()
record("Drop exact duplicate rows", b, len(work))

In [28]:
b = len(work)
cancel_mask = work["Invoice"].str.startswith("C")
returns = work[cancel_mask].copy()
work = work[~cancel_mask]
record("Move cancellation invoices to returns table", b, len(work))

In [29]:
b = len(work)
bad_mask = (work["Quantity"] <= 0) | (work["Price"] <= 0)
adjustments = work[bad_mask].copy()
work = work[~bad_mask]
record("Move Quantity<=0 or Price<=0 rows to adjustments table", b, len(work))

In [30]:
b = len(work)
np_mask = ~work["StockCode"].str.match(r"^\d{5}")
non_products = work[np_mask].copy()
work = work[~np_mask]
record("Move non-product stock codes to non_products table", b, len(work))

In [31]:
b = len(work)
work["Description"] = work["Description"].str.upper().str.strip()
best_desc = (work.dropna(subset=["Description"])
                 .groupby("StockCode")["Description"]
                 .agg(lambda s: s.mode().iloc[0]))
work["Description"] = work["Description"].fillna(work["StockCode"].map(best_desc))
still_missing = work["Description"].isna().sum()
work["Description"] = work["Description"].fillna("UNKNOWN")
record(f"Fill missing descriptions ({still_missing} left as UNKNOWN)", b, len(work))

In [32]:
b = len(work)
work = work.drop_duplicates()
record("Drop duplicates revealed after fixing descriptions", b, len(work))

In [33]:
sales = work.reset_index(drop=True)
decision_log = pd.DataFrame(log)
decision_log

,step,rows_before,rows_after,rows_removed
0,Drop exact duplicate rows,1067371,1033036,34335
1,Move cancellation invoices to returns table,1033036,1013932,19104
2,Move Quantity<=0 or Price<=0 rows to adjustmen...,1013932,1007913,6019
3,Move non-product stock codes to non_products t...,1007913,1003214,4699
4,Fill missing descriptions (0 left as UNKNOWN),1003214,1003214,0
5,Drop duplicates revealed after fixing descript...,1003214,1003214,0


In [34]:
print(f"Started with {len(df):,} rows. Clean sales table: {len(sales):,} rows ({len(sales)/len(df):.1%} kept).")
print(f"Returns: {len(returns):,} | Adjustments: {len(adjustments):,} | Non-products: {len(non_products):,}")

Started with 1,067,371 rows. Clean sales table: 1,003,214 rows (94.0% kept).
Returns: 19,104 | Adjustments: 6,019 | Non-products: 4,699


### 9.1 Validation checks
Automatic tests that confirm the cleaned table follows our rules. If any test fails, the notebook stops with an error.

In [35]:
checks = {
    "No duplicate rows":             sales.duplicated().sum() == 0,
    "No cancellation invoices":      (~sales["Invoice"].str.startswith("C")).all(),
    "All quantities positive":       (sales["Quantity"] > 0).all(),
    "All prices positive":           (sales["Price"] > 0).all(),
    "No missing descriptions":       sales["Description"].notna().all(),
}
for name, passed in checks.items():
    print("PASS" if passed else "FAIL", "-", name)
assert all(checks.values()), "A validation check failed. Review the cleaning steps."

PASS - No duplicate rows
PASS - No cancellation invoices
PASS - All quantities positive
PASS - All prices positive
PASS - No missing descriptions
